# Tempos de Viagem por Caminhada

[descrição]



# Backend

In [ ]:
import datetime as dt
import os
import pathlib
import subprocess

import geopandas as gpd
import h3
import pandas as pd
from pyrosm import get_data
import r5py
from tqdm.auto import tqdm

In [2]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

pd.options.display.float_format = '{:,.2f}'.format

tqdm_bar_format = (
    "{desc:<12} {percentage:3.0f}%|{bar}| "
    "{n_fmt}/{total_fmt} • {rate_fmt} • {elapsed}<{remaining} {postfix}"
)

In [3]:
out_folder = os.environ.get('OUT_FOLDER')
out_folder = pathlib.Path(out_folder)

db_folder = os.environ.get('DB_FOLDER')
db_folder = pathlib.Path(db_folder)

## INPUTS

In [4]:
inpath = out_folder / 'A/pop_weighted_centroids.parquet'

centroids = gpd.read_parquet(inpath)

## osm.pbf

## Routing

In [5]:
ORIGINS_DESTINATIONS = (
    centroids
    .rename(columns={'hex_id': 'id'})
    .to_crs(31983) # just in case...
    .reindex(columns=['id', 'aperture', 'geometry'])
    .to_crs(4326)
    #.query('aperture == 9')
    )

output_pbf = r"C:\Users\brand\OneDrive\Documentos\Coppe\thesis\outputs\A\bh.osm.pbf"

In [ ]:
with tqdm(
    ORIGINS_DESTINATIONS.groupby("aperture"),
    desc="Apertures",
    bar_format=tqdm_bar_format,
    colour="#AA4499"
    ) as outer_bar:
    for aperture, data in outer_bar:
        outer_bar.set_postfix_str(f"Resolution {aperture}")

        outpath = out_folder / "B"
        aperture_dir = outpath / f"travel_times/walk"
        aperture_dir.mkdir(parents=True, exist_ok=True)

        transport_network = r5py.TransportNetwork(
            output_pbf,
            elevation_model=out_folder / 'A/dem_bh.tiff',
        )

        travel_times = r5py.TravelTimeMatrix(
            transport_network,
            origins=data,
            max_time=dt.timedelta(minutes=60),
            transport_modes=[
                r5py.TransportMode.WALK,
            ],
            snap_to_network=np.ceil(
                h3.average_hexagon_edge_length(
                    aperture,
                    unit="m"
                )
            ),
        )

        parquet_file = aperture_dir / f"aperture_{aperture}.parquet"
        travel_times.to_parquet(parquet_file, index=False)


In [7]:
centroids

,hex_id,aperture,weighted_x,weighted_y,geometry
0,89a881345b7ffff,9,"618,773.92","7,807,032.23",POINT (618773.916 7807032.228)
1,89a88136103ffff,9,"617,887.11","7,803,727.28",POINT (617887.107 7803727.275)
2,89a88136107ffff,9,"617,594.79","7,803,871.74",POINT (617594.787 7803871.74)
3,89a8813610bffff,9,"618,186.52","7,803,843.70",POINT (618186.524 7803843.704)
4,89a8813610fffff,9,"617,898.01","7,804,014.35",POINT (617898.014 7804014.346)
...,...,...,...,...,...
166442,8ba88cdb6db2fff,11,"614,178.48","7,792,959.86",POINT (614178.483 7792959.858)
166443,8ba88cdb6db3fff,11,"614,141.88","7,792,978.34",POINT (614141.88 7792978.343)
166444,8ba88cdb6db4fff,11,"614,138.92","7,792,877.22",POINT (614138.918 7792877.223)
166445,8ba88cdb6db5fff,11,"614,093.98","7,792,904.15",POINT (614093.983 7792904.153)
